# 20 — Phase 6: 최종 발표 데모 (v3)

> **변경 이력 (v3)**
> - 시연 이미지: 5종 → **4종** (live2 제거)
> - 앱: `streamlit_demo_v4.py` 사용 (직접 업로드 시 크롭 과정 시각화 포함)
> - 보강: explain_v2 적용 (spoof_type 후처리 + 이미지 기반 캡션)
>
> **시연 이미지 4종**
> | 번호 | 이미지 | 기대 판정 |
> |------|-------|----------|
> | ① | 본인 Live | REAL ✅ |
> | ② | Print Attack | FAKE 🚨 |
> | ③ | Replay Attack | FAKE 🚨 |
> | ④ | Mask Attack | FAKE 🚨 |
>
> **실행 순서:** Cell 0 → 1 → 2 → 3 (앱 실행)

## Cell 0 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 패키지 설치 및 경로 설정

In [ ]:
!pip install pyngrok streamlit -q
!apt-get install -y fonts-nanum -q

import os, sys, json
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path

BASE       = '/content/drive/MyDrive/face-anti-spoofing'
DEMO_DIR   = f'{BASE}/data/demo_images'
APP_DIR    = f'{BASE}/app'
SRC_DIR    = f'{BASE}/src'
REPORT_DIR = f'{BASE}/reports/phase6'
TEST_DIR   = f'{BASE}/data/my_test_images'

os.makedirs(DEMO_DIR,   exist_ok=True)
os.makedirs(APP_DIR,    exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

print('TF  :', tf.__version__)
print('GPU :', tf.config.list_physical_devices('GPU'))

# demo_meta.json 확인 (21번 노트북 Cell 6 결과)
meta_path = f'{DEMO_DIR}/demo_meta.json'
if os.path.exists(meta_path):
    meta = json.load(open(meta_path, encoding='utf-8'))
    print(f'\n✅ demo_meta.json 확인: {list(meta.keys())}')
    for k, v in meta.items():
        exists = '✅' if os.path.exists(v['dst']) else '❌ 없음'
        print(f'  {k}: {exists} | {v["verdict"]} ({v["spoof_prob"]:.1%})')
else:
    print('\n⚠️ demo_meta.json 없음 → 21번 노트북 Cell 6 먼저 실행하세요')

print('\n✅ 경로 설정 완료')

## Cell 2 — streamlit_demo_v4.py 확인

> `streamlit_demo_v4.py`는 Drive에 수동으로 업로드한 파일을 사용합니다.  
> 경로: `/content/drive/MyDrive/face-anti-spoofing/app/streamlit_demo_v4.py`  
> 파일이 없으면 아래 셀에서 자동 생성합니다.

In [ ]:
APP_PATH = f'{APP_DIR}/streamlit_demo_v4.py'

if os.path.exists(APP_PATH):
    print(f'✅ 앱 파일 확인: {APP_PATH}')
else:
    print(f'⚠️ 앱 파일 없음 → 자동 생성 중...')

    app_code = """
import os, sys, json
import numpy as np
import cv2
import streamlit as st
from pathlib import Path

BASE         = "/content/drive/MyDrive/face-anti-spoofing"
SRC_DIR      = f"{BASE}/src"
DEMO_DIR     = f"{BASE}/data/demo_images"
CAPTION_JSON = f"{BASE}/results/phase4/llava_captions.json"
MODEL_PATH   = f"{BASE}/models/stage2_webcam_v3.h5"
sys.path.insert(0, SRC_DIR)
from xai_explainer import explain

st.set_page_config(
    page_title="Face Anti-Spoofing XAI Demo",
    page_icon="\\U0001f6e1",
    layout="wide",
)

LIVE_MEAN     = {"laplacian": 383.0, "fft_high": 1134.0}
LAP_THRESHOLD = 157

SPOOF_KO = {
    0: "Live (실제 얼굴)",
    1: "Print Attack (인쇄 공격)",
    2: "Replay Attack (화면 재촬영)",
    3: "3D Mask (입체 마스크)",
}

REGION_MAP = {
    "upper-center":"forehead region", "upper-left":"forehead-left",
    "upper-right":"forehead-right",   "mid-center":"nose and cheek area",
    "mid-left":"left cheek",          "mid-right":"right cheek",
    "lower-center":"mouth and chin",  "lower-left":"lower-left jaw",
    "lower-right":"lower-right jaw",  "full-face":"entire face",
    "none":"no concentrated region",
}

@st.cache_resource
def get_model():
    import tensorflow as tf
    return tf.keras.models.load_model(MODEL_PATH, compile=False)

@st.cache_resource
def get_cascade():
    return cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

def detect_heatmap_region(heatmap_raw, threshold=0.5):
    if heatmap_raw is None: return "unknown"
    hm = cv2.resize(heatmap_raw, (9,9))
    active = hm >= threshold
    if active.mean() > 0.6: return "full-face"
    rows = np.where(active.any(axis=1))[0]
    cols = np.where(active.any(axis=0))[0]
    if len(rows) == 0: return "none"
    v = "upper" if rows.mean()<3 else ("lower" if rows.mean()>6 else "mid")
    h = "left"  if cols.mean()<3 else ("right" if cols.mean()>6 else "center")
    return f"{v}-{h}"

def anchor_based_type_correction(spoof_type_idx, spoof_type_probs, anchor_stats):
    if spoof_type_idx not in (1,3): return spoof_type_idx
    p_print = spoof_type_probs.get(1,0)
    p_mask  = spoof_type_probs.get(3,0)
    if abs(p_print-p_mask) >= 0.3: return spoof_type_idx
    return 3 if anchor_stats.get("laplacian",0) > LAP_THRESHOLD else 1

def build_image_caption(verdict, spoof_type_idx, anchor_stats, heatmap_raw):
    lap = anchor_stats.get("laplacian",0)
    fft = anchor_stats.get("fft_high",0)
    region_desc = REGION_MAP.get(detect_heatmap_region(heatmap_raw), "face region")
    lap_desc = (
        f"very low sharpness (Lap={lap:.0f})" if lap<LIVE_MEAN["laplacian"]*0.5 else
        f"reduced sharpness (Lap={lap:.0f})"  if lap<LIVE_MEAN["laplacian"]*0.8 else
        f"high edge contrast (Lap={lap:.0f})" if lap>LIVE_MEAN["laplacian"]*1.2 else
        f"normal sharpness (Lap={lap:.0f})"
    )
    fft_desc = (
        f"low high-freq energy (FFT={fft:.0f})" if fft<LIVE_MEAN["fft_high"]*0.6 else
        f"suppressed high-freq (FFT={fft:.0f})" if fft<LIVE_MEAN["fft_high"]*0.85 else
        f"normal high-freq energy (FFT={fft:.0f})"
    )
    type_hints = {
        0: "No spoofing artifacts — classified as live face.",
        1: "Paper-based attack: flat texture and ink dot pattern visible.",
        2: "Screen replay: digital display interference pattern observed.",
        3: "3D mask: rigid boundary and synthetic texture inconsistency.",
    }
    return (f"Model focused on {region_desc}. "
            f"Texture: {lap_desc}, {fft_desc}. "
            f"{type_hints.get(spoof_type_idx,'')}")

def explain_v2(img_bgr, img_path_str, thr=0.75):
    r = explain(img_bgr, img_path=img_path_str, threshold=thr)
    _model = get_model()
    inp = np.expand_dims(
        cv2.resize(cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB),(224,224)).astype("float32")/255.0, 0
    )
    preds = _model.predict(inp, verbose=0)
    spoof_type_probs = ({i:float(p) for i,p in enumerate(preds[1][0])}
                        if isinstance(preds,list) and len(preds)>=2 else {})
    corrected = anchor_based_type_correction(r["spoof_type_idx"], spoof_type_probs, r["anchor_stats"])
    r["spoof_type_idx"]  = corrected
    r["spoof_type_name"] = SPOOF_KO.get(corrected, r["spoof_type_name"])
    if r["verdict"] == "REAL":
        r["spoof_type_name"] = "Live (실제 얼굴)"
    r["llava_caption"] = build_image_caption(
        r["verdict"], r["spoof_type_idx"], r["anchor_stats"], r["heatmap_raw"]
    )
    return r

def crop_face(img_bgr, target_size=224, margin_ratio=0.3):
    cascade = get_cascade()
    gray  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    if len(faces) == 0:
        faces = cascade.detectMultiScale(gray, 1.05, 3, minSize=(40,40))
    if len(faces) == 0:
        h,w = img_bgr.shape[:2]; s = min(h,w)
        return cv2.resize(img_bgr[(h-s)//2:(h+s)//2,(w-s)//2:(w+s)//2],(target_size,target_size)), False
    x,y,w,h = max(faces, key=lambda f:f[2]*f[3])
    ih,iw = img_bgr.shape[:2]
    mx,my = int(w*margin_ratio), int(h*margin_ratio)
    x1,y1 = max(0,x-mx), max(0,y-my)
    x2,y2 = min(iw,x+w+mx), min(ih,y+h+my)
    return cv2.resize(img_bgr[y1:y2,x1:x2],(target_size,target_size)), True

# ── 헤더 ─────────────────────────────────────────────────────
st.title("\\U0001f6e1 Face Anti-Spoofing — 3-Layer XAI Demo")
st.caption("멀티태스크 MobileNetV2 + Grad-CAM + 수치 앵커링 + 이미지 기반 자연어 설명")

meta_path = f"{DEMO_DIR}/demo_meta.json"
demo_meta = json.load(open(meta_path,encoding="utf-8")) if os.path.exists(meta_path) else {}

LABELS = {
    "01_webcam_live" : "① 본인 Live — REAL 기대",
    "02_print_attack": "② Print Attack — FAKE 기대",
    "03_replay_attack":"③ Replay Attack — FAKE 기대",
    "04_mask_attack"  :"④ Mask Attack — FAKE 기대",
    "upload"          :"📤 직접 업로드 (크롭 포함)",
}

# ── 사이드바 ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ 설정")
    thr = st.slider("판정 임계값", 0.50, 0.99, 0.75, 0.01)
    st.divider()
    st.subheader("🎬 시연 이미지")
    mode = st.radio("입력 방식", list(LABELS.values()), index=0)
    st.divider()
    st.subheader("📊 FAR 대시보드")
    import pandas as pd
    st.dataframe(pd.DataFrame({
        "공격 유형":["Print","Replay","Mask","전체"],
        "FAR (%)":[1.33,0.00,0.00,0.44],
        "목표치 (%)":[5,10,8,5],
    }).set_index("공격 유형"), use_container_width=True)

# ── 이미지 로드 ───────────────────────────────────────────────
img_bgr, cur_path = None, ""

if mode == "📤 직접 업로드 (크롭 포함)":
    st.subheader("📤 이미지 업로드")
    st.caption("얼굴이 잘 보이는 사진을 업로드하면 자동으로 크롭 후 분석합니다.")
    uploaded_file = st.file_uploader(
        "얼굴 이미지 선택 (.jpg / .png)",
        type=["jpg","jpeg","png"],
        label_visibility="collapsed",
    )
    if uploaded_file:
        arr = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
        img_raw = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img_raw is None:
            st.error("❌ 이미지 디코딩 실패")
        else:
            col1, col2, col3 = st.columns([4, 1, 4])
            with col1:
                st.markdown("**① 업로드 원본**")
                st.image(
                    cv2.cvtColor(cv2.resize(img_raw,(224,224)), cv2.COLOR_BGR2RGB),
                    caption=f"{img_raw.shape[1]}×{img_raw.shape[0]}",
                    use_container_width=True,
                )
            with st.spinner("얼굴 감지 중..."):
                img_cropped, detected = crop_face(img_raw)
            with col2:
                st.markdown("<br><br><br>", unsafe_allow_html=True)
                st.markdown("### →")
            with col3:
                st.markdown("**② 얼굴 크롭 (224×224)**")
                status = "✅ 얼굴 감지 성공" if detected else "⚠️ center crop fallback"
                st.image(
                    cv2.cvtColor(img_cropped, cv2.COLOR_BGR2RGB),
                    caption=status,
                    use_container_width=True,
                )
            st.markdown("**③ XAI 분석 결과** ↓")
            st.divider()
            img_bgr  = img_cropped
            cur_path = uploaded_file.name
    else:
        st.info("사이드바에서 이미지를 업로드해주세요.")
else:
    key_map = {v:k for k,v in LABELS.items()}
    key = key_map.get(mode)
    if key and key in demo_meta:
        cur_path = demo_meta[key]["dst"]
        img_bgr  = cv2.imread(cur_path)
        if img_bgr is None:
            st.error(f"이미지 로드 실패: {cur_path}")
    else:
        st.warning("시연 이미지가 없습니다. 21번 노트북 Cell 6 먼저 실행하세요.")

# ── XAI 결과 ─────────────────────────────────────────────────
if img_bgr is not None:
    with st.spinner("🔍 분석 중..."):
        r = explain_v2(img_bgr, cur_path, thr)

    verdict = r["verdict"]
    prob    = r["spoof_prob"]
    stype   = r["spoof_type_name"]
    caption = r.get("llava_caption") or "(캡션 없음)"

    if verdict == "REAL":
        st.success(f"✅ REAL (Live) — spoof_prob: {prob:.1%}  |  threshold: {thr:.2f}")
    else:
        st.error(f"🚨 FAKE ({stype}) — spoof_prob: {prob:.1%}  |  threshold: {thr:.2f}")

    c1, c2, c3 = st.columns(3)
    with c1:
        st.subheader("📷 원본 이미지")
        st.image(cv2.cvtColor(cv2.resize(img_bgr,(224,224)),cv2.COLOR_BGR2RGB),
                 use_container_width=True)
        st.caption(f"유형: {stype}")
    with c2:
        st.subheader("🔥 Layer 1: Grad-CAM")
        st.image(r["heatmap_overlay"], use_container_width=True)
        region = detect_heatmap_region(r["heatmap_raw"])
        st.caption(f"활성 영역: {REGION_MAP.get(region, region)}")
    with c3:
        st.subheader("📐 Layer 2: 수치 앵커링")
        m1, m2 = st.columns(2)
        m1.metric("Laplacian", f"{r['anchor_stats']['laplacian']:.0f}")
        m2.metric("FFT High",  f"{r['anchor_stats']['fft_high']:.0f}")
        st.info(f"해석: {r['anchor_interp']}")
        st.subheader("💬 Layer 3: 자연어 설명")
        st.write(r.get("xai_text") or caption)
    with st.expander("🔬 상세 수치"):
        st.json({
            "verdict": verdict, "spoof_prob": round(float(prob),4),
            "spoof_type": stype, "threshold": thr,
            "anchor": r["anchor_stats"], "caption": caption,
        })
"""

    with open(APP_PATH, 'w', encoding='utf-8') as f:
        f.write(app_code.strip())
    print(f'✅ 앱 파일 자동 생성: {APP_PATH}')

## Cell 3 — Streamlit 실행 (ngrok)

In [ ]:
import subprocess, time
from pyngrok import ngrok

# ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # ← 최초 1회만

!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(2)

proc = subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/streamlit_demo_v4.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)

tunnel = ngrok.connect(8501)
print('\n' + '='*60)
print('🚀 Streamlit 데모 앱 실행 중')
print(f'   외부 URL: {tunnel.public_url}')
print('='*60)
print('\n발표 시연 순서:')
print('  ① 본인 Live       → REAL ✅')
print('  ② Print Attack    → FAKE 🚨 + 히트맵')
print('  ③ Replay Attack   → FAKE 🚨 + 히트맵')
print('  ④ Mask Attack     → FAKE 🚨 + 히트맵')
print('  ⑤ 직접 업로드     → 업로드→크롭→분석 3단계 시연')

## ✅ 발표 당일 체크리스트

| 항목 | 확인 |
|------|------|
| Colab T4 GPU 런타임 연결 | ⬜ |
| Cell 0: Drive 마운트 | ⬜ |
| Cell 1: demo_meta.json 4종 모두 ✅ | ⬜ |
| Cell 2: 앱 파일 확인 | ⬜ |
| Cell 3: ngrok URL 확인 + 브라우저 열기 | ⬜ |
| 4종 라디오 버튼 순서대로 판정 확인 | ⬜ |
| 직접 업로드: 크롭 → 분석 흐름 확인 | ⬜ |

---

### 사전 조건

| 조건 | 준비 방법 |
|------|----------|
| `demo_meta.json` (4종 이미지 정보) | 21번 노트북 Cell 6 실행 |
| `xai_explainer.py` | 22번 노트북 Cell 8 실행 (v2 교체) |
| `streamlit_demo_v4.py` | Drive에 수동 업로드 또는 Cell 2 자동 생성 |